# Input validation and abuse prevention

**Module 16 — Secure AI Services and APIs**

Most AI abuse does not need a novel attack. It needs an endpoint that accepts a 4 MB string, an unbounded list,
a negative quantity, or ten thousand requests a minute. This notebook validates inputs with Pydantic, applies a
token-bucket limiter and a cost budget, and measures what each control rejects.

## Learning objectives

1. Express request contracts as a Pydantic model with explicit bounds.
2. Reject malformed input at the boundary, before any model runs.
3. Implement a token bucket and a per-user cost budget.
4. Measure rejection rates on a table of abusive inputs.
5. Distinguish validation (correctness) from abuse control (volume and cost).

## Background

Two different jobs, often confused:

* **Validation** answers "is this request well-formed and within the contract". Failure is `422`.
* **Abuse control** answers "should this caller be allowed to consume this much". Failure is `429`.

A schema without limits is a denial-of-service surface: the cheapest way to exhaust a model service is to send
the largest input it will accept, repeatedly. Bounds belong in the schema, and they must be justified by what
the model can actually process.

## Dependencies and installation

```bash
python -m pip install "pydantic>=2.7" "pandas>=2.2"
```

Standard library plus Pydantic. No network, no model, no API keys.

In [1]:
import time
import unicodedata

import pandas as pd
from pydantic import BaseModel, Field, field_validator

pd.set_option("display.width", 150)
print("pydantic ready")

pydantic ready


**Expected output** — `pydantic ready`.

### Step 1 — A contract with limits

Every field is bounded. The text field carries a maximum length, the numeric fields carry ranges, and a
validator normalises Unicode and rejects control characters.

In [2]:
MAX_TEXT = 500


class ClassifyRequest(BaseModel):
    text: str = Field(min_length=1, max_length=MAX_TEXT)
    top_k: int = Field(default=1, ge=1, le=5)
    amount: float = Field(default=0.0, ge=0, le=1000)
    locale: str = Field(default="en", pattern=r"^[a-z]{2}$")

    @field_validator("text")
    @classmethod
    def clean_text(cls, value: str) -> str:
        normalised = unicodedata.normalize("NFKC", value)
        if any(unicodedata.category(ch) == "Cc" for ch in normalised):
            raise ValueError("control characters are not allowed")
        stripped = normalised.strip()
        if not stripped:
            raise ValueError("text is empty after normalisation")
        return stripped


ok = ClassifyRequest(text="App crashes on launch", top_k=2)
print(ok.model_dump())

{'text': 'App crashes on launch', 'top_k': 2, 'amount': 0.0, 'locale': 'en'}


**Expected output** — the validated model as a dict, with the text trimmed. Note the contract has four bounded
fields; each bound is a decision someone must defend in review.

### Step 2 — A table of abusive inputs

Nine requests that a real endpoint receives. Each declares the failure it should produce. "Accepted" is a
finding, not a pass.

In [3]:
cases = [
    ("ok", {"text": "Invoice is overdue", "top_k": 1}),
    ("empty", {"text": "   "}),
    ("oversized", {"text": "a" * (MAX_TEXT + 1)}),
    ("control chars", {"text": "hello\x00world"}),
    ("top_k too large", {"text": "hello", "top_k": 99}),
    ("negative amount", {"text": "refund", "amount": -5}),
    ("bad locale", {"text": "hello", "locale": "EN-gb"}),
    ("wrong type", {"text": 12345}),
    ("sql-ish payload", {"text": "'; DROP TABLE users; --"}),
]

rows = []
for name, payload in cases:
    try:
        model = ClassifyRequest(**payload)
        rows.append({"case": name, "accepted": True, "detail": f"text len {len(model.text)}"})
    except Exception as exc:
        first = str(exc).splitlines()[0][:90]
        rows.append({"case": name, "accepted": False, "detail": first})
validation = pd.DataFrame(rows)
validation

,case,accepted,detail
0,ok,True,text len 18
1,empty,False,1 validation error for ClassifyRequest
2,oversized,False,1 validation error for ClassifyRequest
3,control chars,False,1 validation error for ClassifyRequest
4,top_k too large,False,1 validation error for ClassifyRequest
5,negative amount,False,1 validation error for ClassifyRequest
6,bad locale,False,1 validation error for ClassifyRequest
7,wrong type,False,1 validation error for ClassifyRequest
8,sql-ish payload,True,text len 23


**Expected output** — seven rejected, and the SQL-ish payload **accepted**: it is a perfectly valid string, and
validation is not injection defence. That row is the important one; parameterised queries are the control, not
the schema.

### Step 3 — A token bucket

A bucket refills at a fixed rate and holds a burst. It limits sustained volume while allowing short spikes,
which is what users actually do.

In [4]:
class TokenBucket:
    def __init__(self, capacity=5, refill_per_second=1.0):
        self.capacity = capacity
        self.tokens = capacity
        self.rate = refill_per_second
        self.updated = time.monotonic()

    def allow(self, cost=1.0):
        now = time.monotonic()
        self.tokens = min(self.capacity, self.tokens + (now - self.updated) * self.rate)
        self.updated = now
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False


bucket = TokenBucket(capacity=5, refill_per_second=0.0)   # no refill, so the test is deterministic
decisions = [bucket.allow() for _ in range(8)]
print("first 8 calls:", decisions)
print("accepted:", sum(decisions), "rejected:", decisions.count(False))
assert sum(decisions) == 5

first 8 calls: [True, True, True, True, True, False, False, False]
accepted: 5 rejected: 3


**Expected output** — five accepted then three rejected. Setting the refill rate to zero makes the test
deterministic; in production the same object refills as time passes, and the assertion would be flaky by
design.

### Step 4 — A cost budget, because tokens are not equal

One long request costs more than one short one. Price requests by size so that a caller cannot spend a day's
budget in an hour by sending maximum-length inputs.

In [5]:
def request_cost(model: ClassifyRequest) -> float:
    return 1.0 + len(model.text) / MAX_TEXT * 4.0    # 1 to 5 units


budget_per_hour = 60.0
spent = 0.0
accepted, rejected = 0, 0
for _ in range(40):
    req = ClassifyRequest(text="x" * MAX_TEXT)
    cost = request_cost(req)
    if spent + cost > budget_per_hour:
        rejected += 1
        continue
    spent += cost
    accepted += 1

print(f"accepted {accepted} max-length requests, rejected {rejected}, spent {spent:.1f}/{budget_per_hour}")
assert spent <= budget_per_hour

accepted 12 max-length requests, rejected 28, spent 60.0/60.0


**Expected output** — 12 accepted, 28 rejected, spending 60 of 60 units. Forty requests of 5 units each cannot
fit in a 60-unit budget, and the limiter says so before the model runs. Note the cost function is a policy
choice: justify the multiplier in a capacity review.

### Step 5 — What each layer catches

The summary table is the deliverable: validation, rate limiting and budget cover different failures, and none
covers injection or model misuse.

In [6]:
coverage = pd.DataFrame(
    [
        {"control": "schema bounds", "catches": "oversized, empty, out-of-range, wrong type",
         "does not catch": "semantic abuse, injection"},
        {"control": "unicode normalisation", "catches": "confusable and zero-width tricks",
         "does not catch": "intent"},
        {"control": "token bucket", "catches": "bursts and sustained floods", "does not catch": "distributed sources"},
        {"control": "cost budget", "catches": "expensive-input abuse", "does not catch": "many small cheap requests"},
        {"control": "parameterised queries", "catches": "SQL injection", "does not catch": "prompt injection"},
    ]
)
coverage

,control,catches,does not catch
0,schema bounds,"oversized, empty, out-of-range, wrong type","semantic abuse, injection"
1,unicode normalisation,confusable and zero-width tricks,intent
2,token bucket,bursts and sustained floods,distributed sources
3,cost budget,expensive-input abuse,many small cheap requests
4,parameterised queries,SQL injection,prompt injection


**Expected output** — five rows, each naming a limit. A control that claims to catch everything is a control
nobody tested.

## Exercises

1. **Add a per-tenant bucket.** Give each tenant its own bucket and a global one. Which is checked first, and
   why does the order matter under attack?
2. **Make cost adversarial.** Find an input that passes validation and maximises cost. Propose a schema change
   that removes it.
3. **Test the refill.** With `refill_per_second=1`, send a request every 0.2 s for two seconds and report the
   accept/reject pattern. Use a fake clock so the test is deterministic.
4. **Reject the SQL payload.** Add a validation rule that does — then explain why adding rules like this for
   injection is the wrong layer.

## Reflection questions

1. Why must the maximum input length be justified by the model's capability, not by what the network allows?
2. A 429 and a 422 mean different things to a client. Why does the distinction matter operationally?
3. Which control in this notebook would have prevented the extraction attack in Module 10, and only partially?

## References

1. Pydantic documentation — models, fields and validators.
2. OWASP, *Application Security Verification Standard* 4.0.3, V5 (validation) and V11 (business logic).
3. NIST, *SP 800-63B*, digital identity guidelines — rate limiting and abuse.
4. OWASP, *Top 10 for LLM Applications*, 2025 — LLM10 (unbounded consumption).